In [0]:
spark.sql("""
    CREATE CATALOG IF NOT EXISTS commercelake
""")

display(spark.sql("SHOW CATALOGS"))

catalog
commercelake
samples
system
workspace


In [0]:
CATALOG = "commercelake"

schemas = [
    "landing",
    "bronze",
    "silver",
    "gold",
    "quarantine",
    "monitoring"
]

for schema in schemas:
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}
    """)
    print(f"Schema ready: {CATALOG}.{schema}")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

Schema ready: commercelake.landing
Schema ready: commercelake.bronze
Schema ready: commercelake.silver
Schema ready: commercelake.gold
Schema ready: commercelake.quarantine
Schema ready: commercelake.monitoring


databaseName
bronze
default
gold
information_schema
landing
monitoring
quarantine
silver


In [0]:
CATALOG = "commercelake"
SCHEMA = "landing"
VOLUME = "source_data"

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS
    {CATALOG}.{SCHEMA}.{VOLUME}
    COMMENT 'Raw source files for CommerceLake data ingestion'
""")

print("Managed volume created successfully")

display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA}"))

Managed volume created successfully


database,volume_name
landing,source_data


In [0]:
BASE_PATH = "/Volumes/commercelake/landing/source_data"

source_directories = [
    "customers",
    "products",
    "orders",
    "order_items",
    "payments"
]

for directory in source_directories:
    path = f"{BASE_PATH}/{directory}"
    dbutils.fs.mkdirs(path)
    print(f"Directory ready: {path}")

print("\nSource directories created successfully")

display(dbutils.fs.ls(BASE_PATH))

Directory ready: /Volumes/commercelake/landing/source_data/customers
Directory ready: /Volumes/commercelake/landing/source_data/products
Directory ready: /Volumes/commercelake/landing/source_data/orders
Directory ready: /Volumes/commercelake/landing/source_data/order_items
Directory ready: /Volumes/commercelake/landing/source_data/payments

Source directories created successfully


path,name,size,modificationTime
dbfs:/Volumes/commercelake/landing/source_data/customers/,customers/,0,1795523327122
dbfs:/Volumes/commercelake/landing/source_data/order_items/,order_items/,0,1795523327122
dbfs:/Volumes/commercelake/landing/source_data/orders/,orders/,0,1795523327122
dbfs:/Volumes/commercelake/landing/source_data/payments/,payments/,0,1795523327122
dbfs:/Volumes/commercelake/landing/source_data/products/,products/,0,1795523327123


In [0]:
BASE_PATH = "/Volumes/commercelake/landing/source_data"

TEST_FILE = f"{BASE_PATH}/_storage_validation.txt"

TEST_CONTENT = "CommerceLake storage validation successful"

# 1. Write a test file
dbutils.fs.put(TEST_FILE, TEST_CONTENT, overwrite=True)
print("WRITE TEST: PASSED")

# 2. Read the test file
content = dbutils.fs.head(TEST_FILE)

assert content == TEST_CONTENT, "Content validation failed"

print("READ TEST: PASSED")
print(f"File content: {content}")

# 3. Delete the test file
assert dbutils.fs.rm(TEST_FILE), "File deletion failed"

# 4. Verify deletion
remaining_files = [
    file.name for file in dbutils.fs.ls(BASE_PATH)
]

assert "_storage_validation.txt" not in remaining_files

print("DELETE TEST: PASSED")

print("=" * 50)
print("ALL STORAGE VALIDATION TESTS PASSED")
print("=" * 50)

Wrote 42 bytes.
WRITE TEST: PASSED
READ TEST: PASSED
File content: CommerceLake storage validation successful
DELETE TEST: PASSED
ALL STORAGE VALIDATION TESTS PASSED


In [0]:
TEST_TABLE = "commercelake.monitoring._delta_validation"

# 1. Create a temporary validation table
spark.sql(f"""
    CREATE OR REPLACE TABLE {TEST_TABLE} (
        id INT,
        product_name STRING,
        price DOUBLE
    )
    USING DELTA
""")

print("TABLE CREATION TEST: PASSED")

# 2. Insert sample records
spark.sql(f"""
    INSERT INTO {TEST_TABLE}
    VALUES
        (1, 'Laptop', 75000),
        (2, 'Mobile', 25000),
        (3, 'Keyboard', 2000)
""")

assert spark.table(TEST_TABLE).count() == 3

print("INSERT TEST: PASSED")

# 3. Read records
df = spark.table(TEST_TABLE)

display(df.orderBy("id"))

print("READ TEST: PASSED")

# 4. Update a record
spark.sql(f"""
    UPDATE {TEST_TABLE}
    SET price = 80000
    WHERE id = 1
""")

updated_price = (
    spark.table(TEST_TABLE)
    .filter("id = 1")
    .first()["price"]
)

assert updated_price == 80000

print("UPDATE TEST: PASSED")

# 5. Verify Delta storage format
details = spark.sql(f"DESCRIBE DETAIL {TEST_TABLE}")

assert details.first()["format"] == "delta"

print("DELTA FORMAT TEST: PASSED")

# 6. Display transaction history
history = spark.sql(f"DESCRIBE HISTORY {TEST_TABLE}")

display(
    history.select("version", "timestamp", "operation")
)

print("TRANSACTION HISTORY TEST: PASSED")

print("=" * 50)
print("ALL DELTA LAKE VALIDATION TESTS PASSED")
print("=" * 50)

TABLE CREATION TEST: PASSED
INSERT TEST: PASSED


id,product_name,price
1,Laptop,75000.0
2,Mobile,25000.0
3,Keyboard,2000.0


READ TEST: PASSED
UPDATE TEST: PASSED
DELTA FORMAT TEST: PASSED


version,timestamp,operation
3,2026-11-24T12:29:09.056Z,OPTIMIZE
2,2026-11-24T12:29:06.945Z,UPDATE
1,2026-11-24T12:29:02.634Z,WRITE
0,2026-11-24T12:29:00.956Z,CREATE OR REPLACE TABLE


TRANSACTION HISTORY TEST: PASSED
ALL DELTA LAKE VALIDATION TESTS PASSED


In [0]:
TEST_TABLE = "commercelake.monitoring._delta_validation"

# 1. Read historical version
historical_df = spark.sql(f"""
    SELECT *
    FROM {TEST_TABLE} VERSION AS OF 1
""")

print("HISTORICAL DATA — VERSION 1")

display(historical_df.orderBy("id"))

# 2. Read current version
current_df = spark.table(TEST_TABLE)

print("CURRENT DATA — VERSION 2")

display(current_df.orderBy("id"))

# 3. Validate historical and current values
historical_price = (
    historical_df.filter("id = 1")
    .first()["price"]
)

current_price = (
    current_df.filter("id = 1")
    .first()["price"]
)

assert historical_price == 75000
assert current_price == 80000

print(f"Historical price: {historical_price}")
print(f"Current price: {current_price}")

print("TIME TRAVEL TEST: PASSED")

# 4. Clean up validation table
spark.sql(f"DROP TABLE IF EXISTS {TEST_TABLE}")

assert not spark.catalog.tableExists(TEST_TABLE)

print("CLEANUP TEST: PASSED")

print("=" * 50)
print("ALL TIME TRAVEL VALIDATION TESTS PASSED")
print("=" * 50)

HISTORICAL DATA — VERSION 1


id,product_name,price
1,Laptop,75000.0
2,Mobile,25000.0
3,Keyboard,2000.0


CURRENT DATA — VERSION 2


id,product_name,price
1,Laptop,80000.0
2,Mobile,25000.0
3,Keyboard,2000.0


Historical price: 75000.0
Current price: 80000.0
TIME TRAVEL TEST: PASSED
CLEANUP TEST: PASSED
ALL TIME TRAVEL VALIDATION TESTS PASSED


In [0]:
display(spark.sql("SHOW TABLES IN commercelake.monitoring"))

database,tableName,isTemporary
